In [0]:
from pyspark.sql.functions import col, sum

data = [
("Chennai", "Laptop", 1, 75000),
("Chennai", "Mouse", 2, 600),
("Mumbai", "Monitor", 1, 12000),
("Hyderabad", "Keyboard", 3, 1500)
]

columns = ["city", "product", "quantity", "unit_price"]

df = spark.createDataFrame(data, columns)

# 1. 
df = df.withColumn("revenue", col("quantity") * col("unit_price"))

# 2 & 3
result = df.groupBy("city").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

# 4 
result.orderBy(col("total_revenue").desc()).show()

+---------+-------------+--------------+
|     city|total_revenue|total_quantity|
+---------+-------------+--------------+
|  Chennai|        76200|             3|
|   Mumbai|        12000|             1|
|Hyderabad|         4500|             3|
+---------+-------------+--------------+



In [0]:
from pyspark.sql.functions import col, avg, when

data = [
(101, "Arjun Kumar", "Python", 86, 92),
(102, "Neha Gupta", "Python", 74, 80),
(103, "Rahul Nair", "Data Engineering", 88, 85),
(104, "Priya Sharma", "AI", 91, 95)
]

columns = ["student_id", "student_name", "course", "marks", "attendance"]

df = spark.createDataFrame(data, columns)

# Grade column
df = df.withColumn("grade",
    when(col("marks") >= 90, "A")
    .when(col("marks") >= 75, "B")
    .when(col("marks") >= 60, "C")
    .otherwise("Needs Improvement")
)

# Certification status
df = df.withColumn("certification_status",
    when((col("marks") >= 75) & (col("attendance") >= 80), "Eligible")
    .otherwise("Not Eligible")
)

df.show()

# Avg marks per course
df.groupBy("course").agg(avg("marks")).show()

# Eligible count
df.filter(col("certification_status") == "Eligible") \
  .groupBy("course").count().show()

+----------+------------+----------------+-----+----------+-----+--------------------+
|student_id|student_name|          course|marks|attendance|grade|certification_status|
+----------+------------+----------------+-----+----------+-----+--------------------+
|       101| Arjun Kumar|          Python|   86|        92|    B|            Eligible|
|       102|  Neha Gupta|          Python|   74|        80|    C|        Not Eligible|
|       103|  Rahul Nair|Data Engineering|   88|        85|    B|            Eligible|
|       104|Priya Sharma|              AI|   91|        95|    A|            Eligible|
+----------+------------+----------------+-----+----------+-----+--------------------+

+----------------+----------+
|          course|avg(marks)|
+----------------+----------+
|          Python|      80.0|
|Data Engineering|      88.0|
|              AI|      91.0|
+----------------+----------+

+----------------+-----+
|          course|count|
+----------------+-----+
|          Python

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank, avg, when

data = [
(1, "Ravi Kumar", "Engineering", "Hyderabad", 92000),
(2, "Sneha Reddy", "Engineering", "Bengaluru", 97000),
(3, "Amit Shah", "Finance", "Mumbai", 88000),
(4, "Pooja Nair", "HR", "Chennai", 69000)
]

columns = ["emp_id", "emp_name", "department", "city", "salary"]

df = spark.createDataFrame(data, columns)


windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())


df = df.withColumn("rank", rank().over(windowSpec))


df = df.withColumn("salary_level",
    when(col("salary") >= 95000, "Very High")
    .when(col("salary") >= 85000, "High")
    .otherwise("Standard")
)

df.show()


df.filter(col("rank") <= 2).show()


df.groupBy("department").agg(avg("salary")).show()

+------+-----------+-----------+---------+------+----+------------+
|emp_id|   emp_name| department|     city|salary|rank|salary_level|
+------+-----------+-----------+---------+------+----+------------+
|     2|Sneha Reddy|Engineering|Bengaluru| 97000|   1|   Very High|
|     1| Ravi Kumar|Engineering|Hyderabad| 92000|   2|        High|
|     3|  Amit Shah|    Finance|   Mumbai| 88000|   1|        High|
|     4| Pooja Nair|         HR|  Chennai| 69000|   1|    Standard|
+------+-----------+-----------+---------+------+----+------------+

+------+-----------+-----------+---------+------+----+------------+
|emp_id|   emp_name| department|     city|salary|rank|salary_level|
+------+-----------+-----------+---------+------+----+------------+
|     2|Sneha Reddy|Engineering|Bengaluru| 97000|   1|   Very High|
|     1| Ravi Kumar|Engineering|Hyderabad| 92000|   2|        High|
|     3|  Amit Shah|    Finance|   Mumbai| 88000|   1|        High|
|     4| Pooja Nair|         HR|  Chennai| 6900